## **Dataset Preparation (IMDB from TensorFlow)**

In [ ]:
import os
import shutil
import re
import numpy as np
import math
import matplotlib.pyplot as plt
from collections import Counter
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, utils, regularizers
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

In [ ]:
# ====== CONFIG ======
VOCAB_SIZE = 20000
SEQUENCE_LENGTH = 250
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.1
SAMPLE_EDA = 5000  # how many raw examples to sample for heavier EDA
SEED = 42

In [ ]:
# ====== 0. (Optional) Fix corrupted TFDS cache if present ======
# This removes the broken raw copy so it will be freshly downloaded.
# Adjust path if you customized TFDS data_dir via TFDS_CONFIG.
tfds_root = os.path.expanduser("~/tensorflow_datasets")
bad_variant = os.path.join(
    tfds_root, "imdb_reviews", "plain_text", "1.0.0"
)
if os.path.exists(bad_variant):
    info_path = os.path.join(bad_variant, "dataset_info.json")
    if not os.path.exists(info_path):
        print(f"[INFO] Detected missing dataset_info.json; deleting {bad_variant} to force redownload.")
        shutil.rmtree(bad_variant)

In [ ]:
# ====== 1. Load raw IMDB dataset (text + label) ======
# as_supervised=True gives (text, label) pairs
(train_raw, test_raw), ds_info = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    with_info=True,
    as_supervised=True,
)

In [ ]:
# ====== 2. Prepare TextVectorization (tokenize + numericalize + pad) ======
vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,  # pad/truncate here
)

# Adapt on training text only
train_text_only = train_raw.map(lambda text, label: text)
vectorizer.adapt(train_text_only)

# Helper to invert tokenization for inspection
vocab = vectorizer.get_vocabulary()  # index -> token
# Note: vocab[0] is usually '' (padding token if mask_zero=False), depending on configuration.

def decode_vectorized(sequence):
    """Map integer sequence back to tokens (approximate original words)."""
    return " ".join(vocab[i] for i in sequence if i != 0)

In [ ]:
# ====== 3. Preprocess datasets (vectorize + batch) ======
def preprocess(text, label):
    return vectorizer(text), label

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(10000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# ====== 4. Extract raw samples for EDA ======
sample_raw = list(train_raw.take(SAMPLE_EDA))
texts = [text.numpy().decode("utf-8") for text, _ in sample_raw]
labels = [label.numpy() for _, label in sample_raw]

In [ ]:
# ====== 5. EDA: label balance ======
def print_balance(name, lbls):
    c = Counter(lbls)
    total = len(lbls)
    print(f"{name} balance: {dict(c)} (pos {c[1]/total*100:.1f}%, neg {c[0]/total*100:.1f}%)")

print_balance("Sampled training", labels)

In [ ]:
# ====== 6. EDA: review length distribution (word-level) ======
def tokenize_simple(text):
    return re.findall(r"\b\w+\b", text.lower())

lengths = [len(tokenize_simple(t)) for t in texts]
print(f"Review lengths (words) — mean: {np.mean(lengths):.1f}, median: {np.median(lengths):.1f}, std: {np.std(lengths):.1f}")

plt.figure()
plt.hist(lengths, bins=50)
plt.title("Review Lengths (words, before vectorization)")
plt.xlabel("Length (words)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# ====== 7. EDA: length vs label ======
lengths_pos = [len(tokenize_simple(t)) for t, l in zip(texts, labels) if l == 1]
lengths_neg = [len(tokenize_simple(t)) for t, l in zip(texts, labels) if l == 0]
print(f"Avg length positive review: {np.mean(lengths_pos):.1f}; negative: {np.mean(lengths_neg):.1f}")

# Boxplot + jittered points
plt.figure()
box = plt.boxplot([lengths_pos, lengths_neg], tick_labels=["positive", "negative"], patch_artist=True)

# Jittered raw points
def jittered_x(base, size):
    return base + np.random.normal(loc=0.0, scale=0.08, size=size)  # adjust scale for spread

plt.scatter(jittered_x(1, len(lengths_pos)), lengths_pos, alpha=0.3, s=10, label="positive samples", edgecolors="none")
plt.scatter(jittered_x(2, len(lengths_neg)), lengths_neg, alpha=0.3, s=10, label="negative samples", edgecolors="none")

plt.title("Review Length by Label (word count) with Jitter")
plt.ylabel("Length (words)")
plt.legend(frameon=False, fontsize="small")
plt.tight_layout()
plt.show()

In [ ]:
# ====== 8. EDA: most common tokens (after simple tokenization, exclude basic stopwords) ======
stopwords = {
    "the", "and", "a", "to", "is", "it", "i", "this", "that", "of",
    "in", "was", "for", "with", "as", "but", "movie", "film", "on", "not"
}
token_counter = Counter()
for t in texts:
    tokens = tokenize_simple(t)
    filtered = [tok for tok in tokens if tok not in stopwords]
    token_counter.update(filtered)
top_n = 25
common_tokens = token_counter.most_common(top_n)
print(f"Top {top_n} tokens (excluding basic stopwords):")
for tok, freq in common_tokens:
    print(f"  {tok:12} {freq}")

# bar plot
words, freqs = zip(*common_tokens)
plt.figure()
plt.barh(list(reversed(words)), list(reversed(freqs)))
plt.title(f"Top {top_n} Tokens")
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# ====== 9. EDA: bigram frequency (excluding stopwords) ======
def get_ngrams(texts_list, n=2, stopwords=None, top_k=20):
    c = Counter()
    for t in texts_list:
        tokens = [w for w in tokenize_simple(t) if w not in (stopwords or set())]
        for i in range(len(tokens) - n + 1):
            gram = tuple(tokens[i : i + n])
            c[gram] += 1
    return c.most_common(top_k)

top_bigrams = get_ngrams(texts, n=2, stopwords=stopwords, top_k=20)
print("\nTop 20 bigrams (excluding basic stopwords):")
for gram, freq in top_bigrams:
    print(f"  {' '.join(gram):20} {freq}")

In [ ]:
# ======10. EDA: tokenization -> numerical and padding verification ======
# Take one batch to inspect
for batch_x, batch_y in train_ds.take(1):
    # batch_x shape: (batch_size, SEQUENCE_LENGTH)
    nonzero_counts = tf.reduce_sum(tf.cast(tf.not_equal(batch_x, 0), tf.int32), axis=1)
    print("\nVectorized batch shape:", batch_x.shape)
    print("Nonzero token counts in first 10 examples (post-vectorization):", nonzero_counts.numpy()[:10])
    # Decode one example
    print("\nDecoded vectorized example (first in batch):")
    print(decode_vectorized(batch_x[0].numpy()))
    break

In [ ]:
# ======11. EDA: padding effect distribution ======
nonzero_all = []
for batch_x, _ in train_ds.take(20):  # sample few batches
    counts = tf.reduce_sum(tf.cast(tf.not_equal(batch_x, 0), tf.int32), axis=1).numpy()
    nonzero_all.extend(counts.tolist())

plt.figure()
plt.hist(nonzero_all, bins=50)
plt.title("Non-padding token counts after vectorization/padding")
plt.xlabel("Nonzero tokens per sequence")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
print(f"After vectorization, mean nonzero tokens: {np.mean(nonzero_all):.1f}, max: {np.max(nonzero_all)}")


In [ ]:
# ======12. Show some raw reviews with labels ======
print("\nSample raw reviews with labels (first 4):")
for i in range(4):
    lbl = "positive" if labels[i] == 1 else "negative"
    snippet = texts[i].replace("\n", " ")[:300]
    print(f"[{lbl}] {snippet}...")
    print("-" * 80)

### **Building the RNN Model (using LSTM)**

## **Building the RNN Model**

This section implements a complete RNN model for sentiment analysis using TensorFlow and Keras with the following architecture:
- **Input layer**: Accepts integer sequences of fixed length
- **Embedding layer**: Converts integer tokens to dense vector representations
- **RNN layer**: Uses LSTM for capturing sequential dependencies
- **Fully connected layer**: Dense layer for feature processing
- **Output layer**: Single neuron with sigmoid activation for binary classification

In [ ]:
# ====== RNN Model Architecture Definition ======
def build_rnn_model(
    vocab_size=20000,                    # e.g., 20000
    embedding_dim=128,             # Embedding vector dimension
    rnn_units=64,                  # LSTM units
    dropout_rate=0.3,              # Dropout rate for regularization
    recurrent_dropout=0.2,         # Recurrent dropout for LSTM
    dense_units=32,                # Fully connected layer units
    sequence_length=250           # Input sequence length (int)
):
    """
    Build RNN model with the following architecture:
    1. Input Layer: Accepts integer sequences
    2. Embedding Layer: Maps tokens to dense vectors
    3. LSTM Layer: Captures sequential dependencies
    4. Fully Connected Layer: Additional feature processing
    5. Output Layer: Binary classification with sigmoid
    """

    # ===== 1. INPUT LAYER =====
    inputs = keras.Input(
        shape=(sequence_length,),
        dtype='int32',
        name='input_sequences'
    )

    # ===== 2. EMBEDDING LAYER =====
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True,
        name='embedding_layer'
    )(inputs)

    # ===== 3. RNN LAYER (LSTM) =====
    x = layers.LSTM(
        units=rnn_units,
        dropout=dropout_rate,
        recurrent_dropout=recurrent_dropout,
        return_sequences=False,
        name='lstm_layer'
    )(x)

    # ===== 4. FULLY CONNECTED LAYER =====
    x = layers.Dense(
        units=dense_units,
        activation='relu',
        name='dense_layer'
    )(x)
    x = layers.Dropout(dropout_rate, name='dropout_layer')(x)

    # ===== 5. OUTPUT LAYER =====
    outputs = layers.Dense(
        units=1,
        activation='sigmoid',
        name='output_layer'
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='sentiment_rnn_model')
    return model


In [2]:
# ====== Model Compilation ======
def compile_model(model, learning_rate=0.001):
    """
    Compile the model with appropriate loss function and optimizer
    """
    model.compile(
        # Binary crossentropy for binary classification
        loss='binary_crossentropy',

        # Adam optimizer with custom learning rate
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),

        # Metrics to track during training
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

# ====== Create and Compile the Model ======
print("Building RNN model...")
model = build_rnn_model()

print("Compiling model...")
model = compile_model(model)

print("Model architecture:")
model.summary()

print(f"\nModel details:")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([keras.backend.count_params(w) for w in model.trainable_weights]):,}")
print(f"Non-trainable parameters: {sum([keras.backend.count_params(w) for w in model.non_trainable_weights]):,}")

Building RNN model...


NameError: name 'build_rnn_model' is not defined

Training the Model

In [3]:
# 1. Load & preprocess IMDB (integer encoded)
vocab_size = 20000
sequence_length = 250

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)
x_all = np.concatenate([x_train_full, x_test], axis=0)
y_all = np.concatenate([y_train_full, y_test], axis=0)

# 2. Pad/truncate
X = utils.pad_sequences(x_all, maxlen=sequence_length, padding='post', truncating='post')
y = y_all  # binary labels 0/1

# 3. Split into train / val / test (~80/10/10)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


NameError: name 'train_test_split' is not defined

In [54]:
# 1. Split: train+val/test, then train/val
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
)  # ~80/10/10

# 2. Compute class weights
def get_class_weights(y):
    counts = np.bincount(y.astype(int))
    total = counts.sum()
    neg, pos = counts[0], counts[1]
    weight_for_0 = (1.0 * total) / (2 * neg) if neg > 0 else 1.0
    weight_for_1 = (1.0 * total) / (2 * pos) if pos > 0 else 1.0
    return {0: weight_for_0, 1: weight_for_1}

class_weights = get_class_weights(y_train)

# 3. Build model (reuse your builder)
def build_rnn_model(
    vocab_size,
    embedding_dim=128,
    rnn_units=64,
    dropout_rate=0.3,
    recurrent_dropout=0.2,
    dense_units=32,
    sequence_length=None
):
    inputs = keras.Input(shape=(sequence_length,), dtype='int32', name='input_sequences')
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name='embedding_layer')(inputs)
    x = layers.LSTM(units=rnn_units, dropout=dropout_rate, recurrent_dropout=recurrent_dropout,
                    return_sequences=False, name='lstm_layer')(x)
    x = layers.Dense(units=dense_units, activation='relu', name='dense_layer')(x)
    x = layers.Dropout(dropout_rate, name='dropout_layer')(x)
    outputs = layers.Dense(units=1, activation='sigmoid', name='output_layer')(x)
    return keras.Model(inputs=inputs, outputs=outputs, name='sentiment_rnn_model')

# Adjust vocab_size and sequence_length to your preprocessing
vocab_size = 20000
sequence_length = X.shape[1]

model = build_rnn_model(vocab_size=vocab_size, sequence_length=sequence_length)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

# 4. Callbacks
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
checkpoint = callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True, verbose=1)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=2
)

# --- Evaluation helper ---
def evaluate_and_report(model, X, y, split_name="set"):
    y_prob = model.predict(X, batch_size=64).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y, y_prob)
    except ValueError:
        auc = float('nan')  # if only one class present
    print(f"=== Performance on {split_name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")
    print(f"\nClassification Report ({split_name}):")
    print(classification_report(y, y_pred, digits=4, zero_division=0))
    print("-" * 60)

# 5. Report on train / val / test
evaluate_and_report(model, X_train, y_train, split_name="train")
evaluate_and_report(model, X_val, y_val, split_name="validation")
evaluate_and_report(model, X_test, y_test, split_name="test")

Epoch 1/5


KeyboardInterrupt: 

In [55]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

# === 1. Load & preprocess IMDB data ===
vocab_size = 20000
sequence_length = 250  # pad/truncate length

# Load integer-encoded IMDB dataset
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.imdb.load_data(num_words=vocab_size)

# Combine so we can split into train/val/test (~80/10/10)
X_all = np.concatenate([x_train_full, x_test_full], axis=0)
y_all = np.concatenate([y_train_full, y_test_full], axis=0)

# Pad/truncate sequences
X_all = keras.utils.pad_sequences(
    X_all,
    maxlen=sequence_length,
    padding='post',
    truncating='post'
)

# === 2. Split data: train/val/test ===
# First split off test (~10%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)
# Then split train vs val (~10% of remaining => ~10% overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
)

# === 3. Class weight helper for imbalance ===
def get_class_weights(y):
    counts = np.bincount(y.astype(int))
    total = counts.sum()
    neg, pos = counts[0], counts[1]
    weight_for_0 = (1.0 * total) / (2 * neg) if neg > 0 else 1.0
    weight_for_1 = (1.0 * total) / (2 * pos) if pos > 0 else 1.0
    return {0: weight_for_0, 1: weight_for_1}

class_weights = get_class_weights(y_train)

# === 4. Model builder ===
def build_rnn_model(
    vocab_size,
    embedding_dim=128,
    rnn_units=64,
    dropout_rate=0.3,
    recurrent_dropout=0.2,
    dense_units=32,
    sequence_length=None
):
    inputs = keras.Input(shape=(sequence_length,), dtype='int32', name='input_sequences')
    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name='embedding_layer')(inputs)
    x = layers.LSTM(
        units=rnn_units,
        dropout=dropout_rate,
        recurrent_dropout=recurrent_dropout,
        return_sequences=False,
        name='lstm_layer'
    )(x)
    x = layers.Dense(units=dense_units, activation='relu', name='dense_layer')(x)
    x = layers.Dropout(dropout_rate, name='dropout_layer')(x)
    outputs = layers.Dense(units=1, activation='sigmoid', name='output_layer')(x)
    return keras.Model(inputs=inputs, outputs=outputs, name='sentiment_rnn_model')

# Instantiate & compile
model = build_rnn_model(vocab_size=vocab_size, sequence_length=sequence_length)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

# === 5. Callbacks ===
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)
checkpoint = callbacks.ModelCheckpoint(
    'best_sentiment_rnn.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# === 6. Train ===
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=2
)

# === 7. Evaluation utilities ===
def full_evaluation(model, X, y, split_name="set"):
    y_prob = model.predict(X, batch_size=64).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y, y_prob)
    except ValueError:
        auc = float("nan")
    print(f"\n=== {split_name.upper()} METRICS ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))
    print(f"\nClassification Report ({split_name}):")
    print(classification_report(y, y_pred, digits=4, zero_division=0))
    print("-" * 60)

# === 8. Report performance ===
full_evaluation(model, X_train, y_train, split_name="train")
full_evaluation(model, X_val, y_val, split_name="validation")
full_evaluation(model, X_test, y_test, split_name="test")

# === 9. Plot training history ===
def plot_training_history(history):
    hist = history.history
    epochs = range(1, len(hist['loss']) + 1)

    def single_plot(metric, title):
        plt.figure()
        plt.plot(epochs, hist[metric], label=f"train {metric}")
        plt.plot(epochs, hist[f"val_{metric}"], label=f"val {metric}")
        plt.title(title)
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.legend()
        plt.tight_layout()
        plt.show()

    single_plot('loss', 'Loss')
    single_plot('accuracy', 'Accuracy')
    single_plot('precision', 'Precision')
    single_plot('recall', 'Recall')
    single_plot('auc', 'AUC')

# Call plotting
plot_training_history(history)


Epoch 1/5


KeyboardInterrupt: 

Updated

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,roc_auc_score, classification_report, confusion_matrix)

In [5]:
# === 1. DATA LOADING & EXPLICIT TOKENIZATION ===
vocab_size = 20000
sequence_length = 250  # fixed length after padding/truncating

print("Loading IMDB dataset (integer-encoded)...")
(x_train_int, y_train_full), (x_test_int, y_test_full) = keras.datasets.imdb.load_data(num_words=vocab_size)

# Combine for unified splitting
X_int_all = np.concatenate([x_train_int, x_test_int], axis=0)
y_all = np.concatenate([y_train_full, y_test_full], axis=0)
print(f"Total examples: {X_int_all.shape[0]}, label distribution: {np.bincount(y_all)}")

# Retrieve word index mapping and invert it to decode integers back to words
word_index = keras.datasets.imdb.get_word_index()
index_to_word = {index + 3: word for word, index in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

def decode_review(int_sequence):
    return " ".join(index_to_word.get(i, "?") for i in int_sequence)

# Decode all sequences to raw text strings
print("Decoding integer sequences back to text for explicit tokenization...")
raw_texts = [decode_review(seq) for seq in X_int_all]
labels = y_all  # binary labels

# Split into train/val/test (80/10/10) with stratification
X_temp_texts, X_test_texts, y_temp, y_test = train_test_split(
    raw_texts, labels, test_size=0.1, random_state=42, stratify=labels
)
X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    X_temp_texts, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
)  # ~10% val overall

print(f"Split sizes: train={len(X_train_texts)}, val={len(X_val_texts)}, test={len(X_test_texts)}")

# TextVectorization: tokenizes and converts to integer sequences with fixed length
vectorizer = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    name="text_vectorization"
)
vectorizer.adapt(X_train_texts)

# Vectorize splits
X_train = vectorizer(np.array(X_train_texts))
X_val = vectorizer(np.array(X_val_texts))
X_test = vectorizer(np.array(X_test_texts))

y_train = np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

print("Shapes after vectorization:", X_train.shape, X_val.shape, X_test.shape)

Loading IMDB dataset (integer-encoded)...
Total examples: 50000, label distribution: [25000 25000]
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Decoding integer sequences back to text for explicit tokenization...
Split sizes: train=40000, val=5000, test=5000
Shapes after vectorization: (40000, 250) (5000, 250) (5000, 250)


In [6]:
# === 2. CLASS WEIGHTS ===
def get_class_weights(y):
    counts = np.bincount(y.astype(int))
    total = counts.sum()
    neg, pos = counts[0], counts[1]
    weight_for_0 = (1.0 * total) / (2 * neg) if neg > 0 else 1.0
    weight_for_1 = (1.0 * total) / (2 * pos) if pos > 0 else 1.0
    return {0: weight_for_0, 1: weight_for_1}

class_weights = get_class_weights(y_train)
print("Computed class weights:", class_weights)

Computed class weights: {0: 1.0, 1: 1.0}


In [7]:
#3a
def build_rnn_model(
    vocab_size,
    embedding_dim=128,
    rnn_units=64,
    dense_units=32,
    dropout_rate=0.5,           # you had increased this
    sequence_length=None
):
    inputs = keras.Input(shape=(sequence_length,), dtype='int32', name='input_sequences')

    # Embedding + spatial dropout (regularizes embedding dimensions across time)
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True,
        name='embedding_layer'
    )(inputs)
    x = layers.SpatialDropout1D(0.3, name='spatial_dropout')(x)

    # Regularized LSTM
    x = layers.LSTM(
        units=rnn_units,
        kernel_regularizer=regularizers.l2(1e-4),
        recurrent_regularizer=regularizers.l2(1e-4),
        name='lstm_layer'
    )(x)  # returns last hidden state

    # Regularized Dense + dropout
    x = layers.Dense(
        units=dense_units,
        activation='relu',
        kernel_regularizer=regularizers.l2(1e-4),
        name='dense_layer'
    )(x)
    x = layers.Dropout(dropout_rate, name='dropout_layer')(x)

    outputs = layers.Dense(units=1, activation='sigmoid', name='output_layer')(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name='rnn_imdb_sentiment_regularized')
    return model

model = build_rnn_model(
    vocab_size=vocab_size,
    embedding_dim=128,
    rnn_units=64,
    dense_units=32,
    dropout_rate=0.5,
    sequence_length=sequence_length
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

print("\nModel summary:")
model.summary()


Model summary:


Model: "rnn_imdb_sentiment_regularized"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_sequences (InputLayer)  │ (None, 250)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_layer (Embedding)   │ (None, 250, 128)          │       2,560,000 │ input_sequences[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ spatial_dropout               │ (None, 250, 128)          │               0 │ embedding_layer[0][0]      │
│ (SpatialDropout1D)            │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 250)               │               0 │ input_sequences[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_layer (LSTM)             │ (None, 64)                │          49,408 │ spatial_dropout[0][0],     │
│                               │                           │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_layer (Dense)           │ (None, 32)                │           2,080 │ lstm_layer[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_layer (Dropout)       │ (None, 32)                │               0 │ dense_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output_layer (Dense)          │ (None, 1)                 │              33 │ dropout_layer[0][0]        │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,611,521 (9.96 MB)

 Trainable params: 2,611,521 (9.96 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# === 4. TRAINING ===
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,  # short because only 5 epochs
    restore_best_weights=True,
    verbose=1
)
checkpoint = callbacks.ModelCheckpoint(
    'best_rnn_imdb.keras',  # native Keras format
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=2
)

Epoch 1/10

Epoch 1: val_loss improved from inf to 0.37697, saving model to best_rnn_imdb.keras
625/625 - 67s - 107ms/step - accuracy: 0.8137 - auc: 0.8900 - loss: 0.4333 - precision: 0.7947 - recall: 0.8457 - val_accuracy: 0.8354 - val_auc: 0.9325 - val_loss: 0.3770 - val_precision: 0.7745 - val_recall: 0.9464 - learning_rate: 0.0010
Epoch 2/10


In [ ]:
#5 EVALUATION ===
def evaluate_metrics_full(model, X, y, split_name="set"):
    y_prob = model.predict(X, batch_size=64).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y, y_pred)
    try:
        auc = roc_auc_score(y, y_prob)
    except ValueError:
        auc = float("nan")

    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)

    report_dict = classification_report(y, y_pred, output_dict=True, zero_division=0, digits=4)
    precision_macro = report_dict["macro avg"]["precision"]
    recall_macro = report_dict["macro avg"]["recall"]
    f1_macro = report_dict["macro avg"]["f1-score"]
    precision_weighted = report_dict["weighted avg"]["precision"]
    recall_weighted = report_dict["weighted avg"]["recall"]
    f1_weighted = report_dict["weighted avg"]["f1-score"]

    cm = confusion_matrix(y, y_pred)
    classif_report_str = classification_report(y, y_pred, digits=4, zero_division=0)

    return {
        "accuracy": acc,
        "auc": auc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "confusion_matrix": cm,
        "classification_report": classif_report_str
    }

def print_summary_table(metrics_dicts):
    headers = [
        "Split", "Acc", "AUC",
        "P", "R", "F1",
        "P(macro)", "R(macro)", "F1(macro)",
        "P(wtd)", "R(wtd)", "F1(wtd)"
    ]
    rows = []
    for split, m in metrics_dicts:
        rows.append([
            split,
            f"{m['accuracy']:.4f}",
            f"{m['auc']:.4f}",
            f"{m['precision']:.4f}",
            f"{m['recall']:.4f}",
            f"{m['f1']:.4f}",
            f"{m['precision_macro']:.4f}",
            f"{m['recall_macro']:.4f}",
            f"{m['f1_macro']:.4f}",
            f"{m['precision_weighted']:.4f}",
            f"{m['recall_weighted']:.4f}",
            f"{m['f1_weighted']:.4f}"
        ])
    col_widths = [max(len(str(cell)) for cell in col) for col in zip(headers, *rows)]
    sep = " | "
    header_line = sep.join(h.ljust(w) for h, w in zip(headers, col_widths))
    divider = "-+-".join("-" * w for w in col_widths)
    print(header_line)
    print(divider)
    for row in rows:
        print(sep.join(c.ljust(w) for c, w in zip(row, col_widths)))

def plot_confusion_matrices_side_by_side(cm_left, cm_right, left_title, right_title, labels=["Negative", "Positive"]):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    disp_left = ConfusionMatrixDisplay(confusion_matrix=cm_left, display_labels=labels)
    disp_left.plot(values_format='d', cmap='Blues', ax=axes[0], colorbar=False)
    axes[0].set_title(left_title)

    disp_right = ConfusionMatrixDisplay(confusion_matrix=cm_right, display_labels=labels)
    disp_right.plot(values_format='d', cmap='Blues', ax=axes[1], colorbar=False)
    axes[1].set_title(right_title)

    for ax in axes.flatten():
        ax.grid(False)
    plt.tight_layout()
    plt.show()

# === Usage: train + validation only ===
train_metrics = evaluate_metrics_full(model, X_train, y_train, split_name="train")
val_metrics = evaluate_metrics_full(model, X_val, y_val, split_name="validation")

print("\n=== Summary Table (Train vs Validation) ===")
print_summary_table([
    ("train", train_metrics),
    ("validation", val_metrics)
])

print("\n--- Train Classification Report ---")
print(train_metrics["classification_report"])
print("\n--- Validation Classification Report ---")
print(val_metrics["classification_report"])

# Confusion matrices: train vs validation
plot_confusion_matrices_side_by_side(
    train_metrics["confusion_matrix"],
    val_metrics["confusion_matrix"],
    left_title="Train Confusion Matrix",
    right_title="Validation Confusion Matrix"
)


In [ ]:
def plot_training_history(history):
    hist = history.history
    # Metrics we want to show (train vs val)
    metric_keys = ['loss', 'accuracy', 'precision', 'recall', 'auc']
    titles = {
        'loss': 'Loss',
        'accuracy': 'Accuracy',
        'precision': 'Precision',
        'recall': 'Recall',
        'auc': 'AUC'
    }
    epochs = range(1, len(hist['loss']) + 1)

    n = len(metric_keys)
    cols = 3
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, key in enumerate(metric_keys):
        ax = axes_flat[i]
        ax.plot(epochs, hist[key], label=f"train {key}")
        ax.plot(epochs, hist[f"val_{key}"], label=f"val {key}")
        ax.set_title(titles.get(key, key))
        ax.set_xlabel("Epoch")
        ax.set_ylabel(key)
        ax.legend()
        ax.grid(True)

    # Hide any unused subplots
    for j in range(n, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.tight_layout()
    plt.show()

plot_training_history(history)

Hyperparameter Tuning

In [ ]:
!pip install -U keras-tuner

In [ ]:
# If you haven't installed keras-tuner yet, uncomment:
# !pip install -U keras-tuner

import numpy as np
import matplotlib.pyplot as plt
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from pprint import pprint

# === 1. Model builder for tuner ===
def build_tunable_model(hp):
    vocab_size = 20000
    sequence_length = 250
    embedding_dim = 128

    # Hyperparameters to tune
    rnn_units = hp.Choice("rnn_units", [32, 64, 96])
    dense_units = hp.Choice("dense_units", [16, 32, 64])
    dropout_rate = hp.Float("dropout_rate", 0.2, 0.6, step=0.1)
    spatial_dropout = hp.Float("spatial_dropout", 0.1, 0.4, step=0.1)
    learning_rate = hp.Choice("learning_rate", [1e-3, 5e-4, 1e-4])
    # Optional extra layer depth (0 or 1 additional dense)
    add_extra_dense = hp.Boolean("add_extra_dense")

    inputs = keras.Input(shape=(sequence_length,), dtype="int32", name="input_sequences")

    x = layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True, name="embedding_layer")(inputs)
    x = layers.SpatialDropout1D(spatial_dropout, name="spatial_dropout")(x)

    x = layers.LSTM(
        units=rnn_units,
        kernel_regularizer=regularizers.l2(1e-4),
        recurrent_regularizer=regularizers.l2(1e-4),
        name="lstm_layer"
    )(x)

    x = layers.Dense(
        units=dense_units,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_layer"
    )(x)
    if add_extra_dense:
        x = layers.Dense(
            units=hp.Choice("extra_dense_units", [16, 32]),
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
            name="extra_dense"
        )(x)
    x = layers.Dropout(dropout_rate, name="dropout_layer")(x)

    outputs = layers.Dense(units=1, activation="sigmoid", name="output_layer")(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name="tunable_rnn")

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0)
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )
    return model

# === 2. Tuner setup ===
tuner = kt.BayesianOptimization(
    build_tunable_model,
    objective=kt.Objective("val_auc", direction="max"),
    max_trials=10,
    seed=42,
    directory="hp_tuning",
    project_name="imdb_rnn_bayes"
)

# Callbacks
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6, verbose=1
)

# === 3. Run search ===

tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=2
)

# === 4. Examine best hyperparameters ===
best_hps = tuner.get_best_hyperparameters(num_trials=3)
print("\nTop 3 hyperparameter sets (by validation objective):")
for i, hp in enumerate(best_hps, 1):
    print(f"\n=== Trial {i} ===")
    pprint({
        "rnn_units": hp.get("rnn_units"),
        "dense_units": hp.get("dense_units"),
        "dropout_rate": hp.get("dropout_rate"),
        "spatial_dropout": hp.get("spatial_dropout"),
        "learning_rate": hp.get("learning_rate"),
        "add_extra_dense": hp.get("add_extra_dense"),
        **({"extra_dense_units": hp.get("extra_dense_units")} if hp.get("add_extra_dense") else {})
    })

# === 5. Retrieve best model and evaluate ===
best_model = tuner.get_best_models(1)[0]

def evaluate(model, X, y, threshold=0.5):
    y_prob = model.predict(X, batch_size=64).flatten()
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "auc": roc_auc_score(y, y_prob),
        "confusion_matrix": confusion_matrix(y, y_pred),
        "classification_report": classification_report(y, y_pred, digits=4, zero_division=0)
    }
    return metrics, y_prob

# Validation evaluation (threshold sweep for best F1)
val_metrics_default, y_val_prob = evaluate(best_model, X_val, y_val, threshold=0.5)
# find best threshold on validation
best_thr = 0.5
best_f1 = val_metrics_default["f1"]
for t in np.linspace(0.1, 0.9, 81):
    m, _ = evaluate(best_model, X_val, y_val, threshold=t)
    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        best_thr = t

print(f"\nBest threshold on validation for F1: {best_thr:.3f} (F1={best_f1:.4f})")
val_metrics_tuned, _ = evaluate(best_model, X_val, y_val, threshold=best_thr)

print("\n=== Validation Performance with Tuned Threshold ===")
pprint({k: v for k, v in val_metrics_tuned.items() if k != "confusion_matrix"})
print(val_metrics_tuned["classification_report"])

# Train performance
train_metrics, _ = evaluate(best_model, X_train, y_train, threshold=best_thr)
print("\n=== Train Performance (same threshold) ===")
pprint({k: v for k, v in train_metrics.items() if k != "confusion_matrix"})

# === 6. Documenting results ===
def summarize_results(name, metrics):
    return {
        "stage": name,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "auc": metrics["auc"],
    }

summary = [
    summarize_results("train", train_metrics),
    summarize_results("validation", val_metrics_tuned)
]

print("\n=== Summary Table ===")
for row in summary:
    print(f"{row['stage']:10} | acc {row['accuracy']:.4f} | prec {row['precision']:.4f} | recall {row['recall']:.4f} | f1 {row['f1']:.4f} | auc {row['auc']:.4f}")

# === 7. (Optional) Retrain on train+validation with best HPs then final test evaluation ===
best_hp = best_hps[0]
final_model = build_tunable_model(best_hp)  # rebuild with best hyperparams
# combine train+val
X_combined = np.concatenate([X_train, X_val], axis=0)
y_combined = np.concatenate([y_train, y_val], axis=0)
final_model.fit(
    X_combined, y_combined,
    epochs=5,
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(monitor="loss", patience=2, restore_best_weights=True)],
    verbose=2
)
# Evaluate on validation again for sanity
final_train_metrics, _ = evaluate(final_model, X_combined, y_combined, threshold=best_thr)
print("\n=== Retrained on train+val performance ===")
pprint({k: v for k, v in final_train_metrics.items() if k != "confusion_matrix"})
